## Configuracion parametros

In [0]:

catalog = "certificacion_databricks_proyecto"
schema = "bronze_michael"
volume = "landing_michael"

landing_path = f"/Volumes/{catalog}/{schema}/{volume}/"

checkpoint_path = f"/Volumes/{catalog}/{schema}/{volume}/_checkpoint/bronze_michael_v3/"
schema_path = f"/Volumes/{catalog}/{schema}/{volume}/_schema/bronze_michael_v3/"

## comprobación lectura landing

In [0]:
display(dbutils.fs.ls(landing_path))

path,name,size,modificationTime
dbfs:/Volumes/certificacion_databricks_proyecto/bronze_michael/landing_michael/METHODOLOGY.md,METHODOLOGY.md,971,1791303051000
dbfs:/Volumes/certificacion_databricks_proyecto/bronze_michael/landing_michael/README.md,README.md,2714,1791303051000
dbfs:/Volumes/certificacion_databricks_proyecto/bronze_michael/landing_michael/_checkpoint/,_checkpoint/,0,1791408078813
dbfs:/Volumes/certificacion_databricks_proyecto/bronze_michael/landing_michael/_schema/,_schema/,0,1791408078814
dbfs:/Volumes/certificacion_databricks_proyecto/bronze_michael/landing_michael/data_dictionary.csv,data_dictionary.csv,1310,1791303052000
dbfs:/Volumes/certificacion_databricks_proyecto/bronze_michael/landing_michael/dataset_summary.csv,dataset_summary.csv,301,1791303051000
dbfs:/Volumes/certificacion_databricks_proyecto/bronze_michael/landing_michael/instagram_reels_clean.csv,instagram_reels_clean.csv,41253,1791303051000
dbfs:/Volumes/certificacion_databricks_proyecto/bronze_michael/landing_michael/instagram_reels_clean.parquet,instagram_reels_clean.parquet,37344,1791303051000
dbfs:/Volumes/certificacion_databricks_proyecto/bronze_michael/landing_michael/missing_value_report.csv,missing_value_report.csv,304,1791303051000
dbfs:/Volumes/certificacion_databricks_proyecto/bronze_michael/landing_michael/niche_coverage_report.csv,niche_coverage_report.csv,266,1791303051000


## Leer autoloader dataset y escribir formato delta

In [0]:
df_bronze = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaEvolutionMode", "rescue")
        .option("cloudFiles.schemaLocation", schema_path)
        .option("cloudFiles.includeExistingFiles", "true")
        .option("pathGlobFilter", "instagram_reels_clean.csv")
        .option("header", "true")
        .option("multiLine", "true")
        .option("quote", '"')
        .option("escape", '"')
        .load(landing_path)
)

In [0]:
df_bronze.printSchema()

root
 |-- post_id: string (nullable = true)
 |-- shortcode: string (nullable = true)
 |-- creator_handle: string (nullable = true)
 |-- caption: string (nullable = true)
 |-- timestamp_utc: string (nullable = true)
 |-- video_duration_sec: string (nullable = true)
 |-- views: string (nullable = true)
 |-- likes: string (nullable = true)
 |-- comments: string (nullable = true)
 |-- caption_word_count: string (nullable = true)
 |-- hashtag_count: string (nullable = true)
 |-- niche: string (nullable = true)
 |-- engagement_rate_pct: string (nullable = true)
 |-- virality_score: string (nullable = true)
 |-- virality_tier: string (nullable = true)
 |-- _rescued_data: string (nullable = true)



In [0]:
print(df_bronze.columns)

['post_id', 'shortcode', 'creator_handle', 'caption', 'timestamp_utc', 'video_duration_sec', 'views', 'likes', 'comments', 'caption_word_count', 'hashtag_count', 'niche', 'engagement_rate_pct', 'virality_score', 'virality_tier', '_rescued_data']


In [0]:
(
    df_bronze.writeStream
        .format("delta")
        .option("checkpointLocation", checkpoint_path)
        .outputMode("append")
        .trigger(availableNow=True)
        .toTable(
            "certificacion_databricks_proyecto.bronze_michael.bronze_michael"
        )
)

## Exploracion dataset y comprobacion 

In [0]:
display(
    spark.table(
        f"{catalog}.{schema}.bronze_michael"
    )
)

post_id,shortcode,creator_handle,caption,timestamp_utc,video_duration_sec,views,likes,comments,caption_word_count,hashtag_count,niche,engagement_rate_pct,virality_score,virality_tier,_rescued_data
3971607901016940165,Dcd_FfvP96F,techburner,Robot Phone🤯 #robot #ai #smartphone #reels,2026-08-25 15:44:22+00:00,51.4,3198411,224480,1073,2,4,Technology & Gadgets,7.05,88.2,Viral Hit,null
3968684083096105119,DcTmSWGsLCf,ranveerallahbadia,"This is genuinely the happiest phase of my life, and I finally get to share it with you all. For everyone who has followed this journey for 10 years now, you know what it took to get here. And this is what the other side looks like. Our full Ladakh road trip vlog is up. The Buddhist magic of Leh, the electricity of the Hemis festival, the raw beauty of Khardungla, the stillness of Nubra valley and the pure legend that is Pangong Lake. All packed into one video. Ladakh should genuinely be on everyone’s bucket list. No traffic, no noise, just wind, water and birds. And for once, actual silence in my own head. 🏔️🙏 #travel #BeerBiceps #Ladakh",2026-08-21 14:55:37+00:00,100.7,2969964,284682,583,116,3,Entertainment,9.6,98.1,Viral Hit,null
3921292129996925127,DZrOmqVAGjH,beerbiceps,World Cup begins today for most of us ❤️🇵🇹 . . . #ronaldo #worldcup #portugal,2026-06-17 05:37:39+00:00,50.7,1982315,371478,3853,12,3,Fitness & Lifestyle,18.93,98.1,Viral Hit,null
3957069018352961979,DbqVU6Gocm7,onepercentclubshow,"Everyone thinks owning an Airbnb is easy passive income. Parul Gulati's experience says otherwise. Tomorrow, she joins me to talk about building Malkin Hostel, investing in a Goa property, what went wrong, and what she'd do differently today. Dropping soon.",2026-08-05 14:17:47+00:00,40.7,1606642,47697,304,40,0,Entertainment,2.99,69.2,High Reach,null
3972244393712137742,DcgPzrwP7YO,techburner,IPhone vs Robot Phone🔥 #reels,2026-08-26 12:48:43+00:00,37.2,1588449,168964,350,4,1,Technology & Gadgets,10.66,97.2,Viral Hit,null
3970151839831436644,DcY0BBFhC1k,beerbiceps,"WAIT FOR IT 😂❤️ If you guys haven’t watched the Ladakh vlog yet, go check it out on YouTube. If you haven’t, you should watch it with your friends and family. #Ladakh #BeerBiceps",2026-08-23 15:30:06+00:00,45.2,1385190,87163,422,31,2,Fitness & Lifestyle,6.32,81.9,Viral Hit,null
3320693160942412829,C4VeNYKCqQd,kabitaskitchen,मेरी ज़िंदगी की सबसे बड़ा और यादगार पल 🥹 @narendramodi @mygovindia #nationalcreatoraward #memorableday #bestday #happiness #primeministernarendramodi #kabitaskitchen #reel #reels #reelkarofeelkaro #trendingreels #trendingsong #reels #trending #trend,2024-03-10 13:33:13+00:00,31.6,1378075,130386,1936,11,14,Food & Cooking,9.6,95.0,Viral Hit,null
3969412398414499932,DcWL4umhxBc,beerbiceps,"At the end of everything, what actually remains is experiential intimacy. Experiencing both the grand moments and the mundane ones with your partner, that kind of intimacy is unmatched. ❤️❤️❤️ Super grateful for the experience 🏔️ #Ladakh #intimacy #Relationships #BeerBiceps",2026-08-22 15:00:15+00:00,35.0,1362541,120742,592,36,4,Fitness & Lifestyle,8.9,92.2,Viral Hit,null
3911367908586588946,DZH-GO5KosS,beerbiceps,"My 33rd Birthday was celebrated in Mahakaal, Ujjain. Of course life is not going to be the same from here on. They say life changes at age 33. They say that life changes after one visits Mahakaal! Couldn’t have had a more wholesome birthday. Shot a powerful Ujjain documentary here with @authorshishirk on my birthday. —- Thank you so much to everyone we met in Indore, Ujjain and MP in general! We will be back. Thank you Shivji 🔱❤️🙏 #jaimahakal",2026-06-03 12:58:39+00:00,19.6,1305768,178880,1070,79,1,Fitness & Lifestyle,13.78,96.4,Viral Hit,null
3975950150500380684,DctaZitPHAM,techburner,IPhone vs Robot Phone🔥 #reels,2026-08-31 15:31:59+00:00,35.7,1215340,81946,184,4,1,Technology & Gadgets,6.76,83.2,Viral Hit,null


In [0]:
display(
    spark.sql("""
        SELECT *
        FROM certificacion_databricks_proyecto.bronze_michael.bronze_michael
        LIMIT 20
    """)
)

post_id,shortcode,creator_handle,caption,timestamp_utc,video_duration_sec,views,likes,comments,caption_word_count,hashtag_count,niche,engagement_rate_pct,virality_score,virality_tier,_rescued_data
3971607901016940165,Dcd_FfvP96F,techburner,Robot Phone🤯 #robot #ai #smartphone #reels,2026-08-25 15:44:22+00:00,51.4,3198411,224480,1073,2,4,Technology & Gadgets,7.05,88.2,Viral Hit,null
3968684083096105119,DcTmSWGsLCf,ranveerallahbadia,"This is genuinely the happiest phase of my life, and I finally get to share it with you all. For everyone who has followed this journey for 10 years now, you know what it took to get here. And this is what the other side looks like. Our full Ladakh road trip vlog is up. The Buddhist magic of Leh, the electricity of the Hemis festival, the raw beauty of Khardungla, the stillness of Nubra valley and the pure legend that is Pangong Lake. All packed into one video. Ladakh should genuinely be on everyone’s bucket list. No traffic, no noise, just wind, water and birds. And for once, actual silence in my own head. 🏔️🙏 #travel #BeerBiceps #Ladakh",2026-08-21 14:55:37+00:00,100.7,2969964,284682,583,116,3,Entertainment,9.6,98.1,Viral Hit,null
3921292129996925127,DZrOmqVAGjH,beerbiceps,World Cup begins today for most of us ❤️🇵🇹 . . . #ronaldo #worldcup #portugal,2026-06-17 05:37:39+00:00,50.7,1982315,371478,3853,12,3,Fitness & Lifestyle,18.93,98.1,Viral Hit,null
3957069018352961979,DbqVU6Gocm7,onepercentclubshow,"Everyone thinks owning an Airbnb is easy passive income. Parul Gulati's experience says otherwise. Tomorrow, she joins me to talk about building Malkin Hostel, investing in a Goa property, what went wrong, and what she'd do differently today. Dropping soon.",2026-08-05 14:17:47+00:00,40.7,1606642,47697,304,40,0,Entertainment,2.99,69.2,High Reach,null
3972244393712137742,DcgPzrwP7YO,techburner,IPhone vs Robot Phone🔥 #reels,2026-08-26 12:48:43+00:00,37.2,1588449,168964,350,4,1,Technology & Gadgets,10.66,97.2,Viral Hit,null
3970151839831436644,DcY0BBFhC1k,beerbiceps,"WAIT FOR IT 😂❤️ If you guys haven’t watched the Ladakh vlog yet, go check it out on YouTube. If you haven’t, you should watch it with your friends and family. #Ladakh #BeerBiceps",2026-08-23 15:30:06+00:00,45.2,1385190,87163,422,31,2,Fitness & Lifestyle,6.32,81.9,Viral Hit,null
3320693160942412829,C4VeNYKCqQd,kabitaskitchen,मेरी ज़िंदगी की सबसे बड़ा और यादगार पल 🥹 @narendramodi @mygovindia #nationalcreatoraward #memorableday #bestday #happiness #primeministernarendramodi #kabitaskitchen #reel #reels #reelkarofeelkaro #trendingreels #trendingsong #reels #trending #trend,2024-03-10 13:33:13+00:00,31.6,1378075,130386,1936,11,14,Food & Cooking,9.6,95.0,Viral Hit,null
3969412398414499932,DcWL4umhxBc,beerbiceps,"At the end of everything, what actually remains is experiential intimacy. Experiencing both the grand moments and the mundane ones with your partner, that kind of intimacy is unmatched. ❤️❤️❤️ Super grateful for the experience 🏔️ #Ladakh #intimacy #Relationships #BeerBiceps",2026-08-22 15:00:15+00:00,35.0,1362541,120742,592,36,4,Fitness & Lifestyle,8.9,92.2,Viral Hit,null
3911367908586588946,DZH-GO5KosS,beerbiceps,"My 33rd Birthday was celebrated in Mahakaal, Ujjain. Of course life is not going to be the same from here on. They say life changes at age 33. They say that life changes after one visits Mahakaal! Couldn’t have had a more wholesome birthday. Shot a powerful Ujjain documentary here with @authorshishirk on my birthday. —- Thank you so much to everyone we met in Indore, Ujjain and MP in general! We will be back. Thank you Shivji 🔱❤️🙏 #jaimahakal",2026-06-03 12:58:39+00:00,19.6,1305768,178880,1070,79,1,Fitness & Lifestyle,13.78,96.4,Viral Hit,null
3975950150500380684,DctaZitPHAM,techburner,IPhone vs Robot Phone🔥 #reels,2026-08-31 15:31:59+00:00,35.7,1215340,81946,184,4,1,Technology & Gadgets,6.76,83.2,Viral Hit,null


In [0]:
spark.sql("""
    SELECT COUNT(*) AS total_registros
    FROM certificacion_databricks_proyecto.bronze_michael.bronze_michael
""").show()

+---------------+
|total_registros|
+---------------+
|             88|
+---------------+



In [0]:
spark.sql("""
    SELECT COUNT(*) AS registros_con_rescate
    FROM certificacion_databricks_proyecto.bronze_michael.bronze_michael
    WHERE _rescued_data IS NOT NULL
""").show()

+---------------------+
|registros_con_rescate|
+---------------------+
|                    0|
+---------------------+



In [0]:
spark.sql("""
    DESCRIBE certificacion_databricks_proyecto.bronze_michael.bronze_michael
""").show(truncate=False)

+-------------------+---------+-------+
|col_name           |data_type|comment|
+-------------------+---------+-------+
|post_id            |string   |NULL   |
|shortcode          |string   |NULL   |
|creator_handle     |string   |NULL   |
|caption            |string   |NULL   |
|timestamp_utc      |string   |NULL   |
|video_duration_sec |string   |NULL   |
|views              |string   |NULL   |
|likes              |string   |NULL   |
|comments           |string   |NULL   |
|caption_word_count |string   |NULL   |
|hashtag_count      |string   |NULL   |
|niche              |string   |NULL   |
|engagement_rate_pct|string   |NULL   |
|virality_score     |string   |NULL   |
|virality_tier      |string   |NULL   |
|_rescued_data      |string   |NULL   |
+-------------------+---------+-------+

